In [2]:
import pulp
import numpy as np
import joblib

In [3]:
Data_Context = joblib.load('context/data_context.pkl')
Training_Context = joblib.load('context/training_context.pkl')

df_original = Data_Context['df_original']
test_data = Training_Context['test_data']
prediction_test = Training_Context['prediction_test']

In [4]:
hasil_prediksi = df_original.loc[test_data.index, ['SKU']].copy()
hasil_prediksi['Tebakan_Demand'] = np.round(prediction_test).astype(int)

demand_bulan_depan = hasil_prediksi.groupby('SKU')['Tebakan_Demand'].max().to_dict()

harga_satuan = df_original.groupby('SKU')['Average_Unit_Price'].mean().to_dict()

In [5]:
MODAL_RESTOCK = 1000000 
KAPASITAS_GUDANG = 10000
MARGIN_PROFIT = 0.3

In [6]:
model = pulp.LpProblem("Optimasi_Restock_Toko_Elektronik", pulp.LpMaximize)

jumlah_beli = {
    sku: pulp.LpVariable(f"Beli_{sku}", lowBound=10, cat='Integer') 
    for sku in demand_bulan_depan.keys()
}

model += pulp.lpSum([jumlah_beli[sku] * harga_satuan[sku] * MARGIN_PROFIT for sku in demand_bulan_depan.keys()]), "Total_Profit"
model += pulp.lpSum([jumlah_beli[sku] * harga_satuan[sku] for sku in demand_bulan_depan.keys()]) <= MODAL_RESTOCK, "Batas_Modal"
model += pulp.lpSum([jumlah_beli[sku] for sku in demand_bulan_depan.keys()]) <= KAPASITAS_GUDANG, "Batas_Gudang"

for sku in demand_bulan_depan.keys():
    model += jumlah_beli[sku] <= demand_bulan_depan[sku], f"Batas_Demand_{sku}"

model.solve()

1

In [7]:
print(f"Model Status: {pulp.LpStatus[model.status]}")
print(f"Base Budget: ${MODAL_RESTOCK:,.2f}")
print(f"Inventory Capacity: {KAPASITAS_GUDANG} unit\n")

print("Next Month's Restock Recommendations\n")
total_biaya = 0
total_unit = 0

for sku in demand_bulan_depan.keys():
    unit_dibeli = int(jumlah_beli[sku].varValue)
    biaya = unit_dibeli * harga_satuan[sku]
    
    total_biaya += biaya
    total_unit += unit_dibeli
    
    print(f"{sku}: Buy {unit_dibeli} unit (Demand Prediction: {demand_bulan_depan[sku]} unit)")

print(f"\nTotal budget used: ${total_biaya:,.2f}")
print(f"Budget Leftovers: ${MODAL_RESTOCK - total_biaya:,.2f}")
print(f"Total Unit in Inventory: {total_unit} unit")
print(f"Estimated Profit: ${pulp.value(model.objective):,.2f}")

Model Status: Optimal
Base Budget: $1,000,000.00
Inventory Capacity: 10000 unit

Next Month's Restock Recommendations

HDP456: Buy 10 unit (Demand Prediction: 870 unit)
LTP123: Buy 10 unit (Demand Prediction: 854 unit)
SKU1001: Buy 67 unit (Demand Prediction: 592 unit)
SKU1002: Buy 226 unit (Demand Prediction: 597 unit)
SKU1003: Buy 623 unit (Demand Prediction: 655 unit)
SKU1004: Buy 188 unit (Demand Prediction: 636 unit)
SKU1005: Buy 10 unit (Demand Prediction: 577 unit)
SMP234: Buy 10 unit (Demand Prediction: 803 unit)
SWT567: Buy 10 unit (Demand Prediction: 901 unit)
TBL345: Buy 301 unit (Demand Prediction: 976 unit)

Total budget used: $1,000,000.00
Budget Leftovers: $0.00
Total Unit in Inventory: 1455 unit
Estimated Profit: $300,000.00
